In [ ]:
from guardian_help import *

import numpy as np
import pandas as pd
import os
from scipy.io import loadmat


path = "./data/"
# Path to the users folder
path_users = path + "users/"

folders = [name for name in os.listdir(path + "users/") if os.path.isdir(os.path.join(path + "users/", name))]
    
print(folders)

dict_all_users_data = load_all_users_data(path_users)
df_all_users = get_all_users_df(dict_all_users_data)
df_all_users['video_number'] = df_all_users['video_number'].astype(int) 

In [ ]:
# create a table with users as rows and video statistics as columns
# for each user, we'll compute: total videos watched, total duration per video, and number of samples per video

df = df_all_users.copy()

# ensure timestamp is numeric
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')

# compute duration per (user, video_number) as max_timestamp - min_timestamp
video_durations = df.groupby(['user', 'video_number'])['timestamp'].agg(
    duration_ms=lambda x: x.max() - x.min()
).reset_index()

# convert duration to seconds for readability
video_durations['duration_s'] = video_durations['duration_ms'] / 1000.0

# compute number of samples per (user, video_number)
video_samples = df.groupby(['user', 'video_number']).size().reset_index(name='n_samples')

# merge duration and sample count
video_stats = video_samples
# I want a column with the incremental index of the video per user
video_stats['index'] = video_stats.groupby('user').cumcount() + 1
video_stats = video_stats.merge(
    video_durations[['user', 'video_number', 'duration_s']],
    on=['user', 'video_number'],
    how='left'
)

# pivot: users as rows, video_number as columns, values = duration in seconds
pivot_duration = video_stats.pivot(index='user', columns='index', values='duration_s')

# pivot: users as rows, video_number as columns, values = number of samples
pivot_samples = video_stats.pivot(index='user', columns='index', values='n_samples')

# rename columns to use simpler format: video_1, video_2, etc.
pivot_duration.columns = [f'video_{col}' for col in pivot_duration.columns]
pivot_samples.columns = [f'{col}' for col in pivot_samples.columns]

# combine both pivots
#  pivot_table = pd.concat([pivot_duration, pivot_samples], axis=1)
pivot_table = pivot_samples.copy()

# add a column for total number of videos watched per user
pivot_table['total_videos_watched'] = video_stats.groupby('user')['video_number'].nunique()
pivot_duration['total_videos_watched'] = video_stats.groupby('user')['video_number'].nunique()

# sort by total videos watched (descending) for easier reading
pivot_table = pivot_table.sort_values('total_videos_watched', ascending=False)
pivot_duration = pivot_duration.sort_values('total_videos_watched', ascending=False)
    

print(f"Table shape: {pivot_table.shape} (users x (videos*2) + 1 summary column)")


In [ ]:
# from pivot_table, get the users which presented more than 20000 samples in each video (for all videos, not just the common ones)
filtered_users = pivot_table[
    (pivot_table.drop(columns=['total_videos_watched']) > 18000).all(axis=1)
].index.tolist()
print("Filtered users with >20000 samples in all videos:", filtered_users)
print("Number of filtered users:", len(filtered_users))

In [ ]:
# heatmap of the pivot table showing number of samples per video per user
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
# The user names on the y-axis and video indices on the x-axis (only number)
sns.heatmap(pivot_table.drop(columns=['total_videos_watched']).T, 
            fmt='g', cmap='YlGnBu', cbar_kws={'label': 'Number of Samples'}, 
            yticklabels=True, xticklabels=True)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.title('Number of Samples per Video per User')
plt.xlabel('Video Index')
plt.ylabel('User')
plt.show()
            


In [ ]:
# heatmap of the pivot table showing number of samples per video per user
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
# The user names on the y-axis and video indices on the x-axis (only number)
sns.heatmap(pivot_duration.drop(columns=['total_videos_watched']).T, 
            fmt='g', cmap='YlGnBu', cbar_kws={'label': 'Duration (s)'}, 
            yticklabels=True, xticklabels=True)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.title('Duration per Video per User')
plt.xlabel('Video Index')
plt.ylabel('User')
plt.show()
            
